In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

class SimplePacmanEnv(gym.Env):
    """
    吃豆子环境，支持课程学习：
    - 网格大小 grid_size (默认15)
    - 可选鬼魂 (with_ghost) 和墙壁 (with_walls)
    - 玩家从 (0,0) 开始
    - 随机生成约 60% 的豆子（起点无豆）
    - 吃到豆子得 +1，吃完所有豆子得 +10，每步 -0.1
    - 鬼魂随机移动（如果启用），碰到鬼魂得 -10 并结束
    - 墙壁格不可进入（如果启用）
    - 状态: 4通道 (玩家, 鬼魂, 墙壁, 豆子)，每个通道 grid_size x grid_size
    """
    def __init__(self, grid_size=15, with_ghost=False, with_walls=False, num_ghosts=1):
        super().__init__()
        self.grid_size = grid_size
        self.with_ghost = with_ghost
        self.with_walls = with_walls
        self.num_ghosts = num_ghosts if with_ghost else 0
        self.ghost_collision = False
        # 动作空间: 0上,1下,2左,3右
        self.action_space = spaces.Discrete(4)

        # 观察空间: 4通道网格，每个像素值0或1
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(4, grid_size, grid_size),
            dtype=np.float32
        )

        self.reset()


    def reset(self,seed=None,options=None):
        if seed is not None:
            np.random.seed(seed)
        # 玩家起始位置
        self.player_pos = [0, 0]

        # 随机生成豆子（约60%）
        total_cells = self.grid_size * self.grid_size
        num_dots = int(total_cells * 0.6)
        indices = np.random.choice(total_cells, num_dots, replace=False)
        self.dot_grid = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        self.dot_grid.flat[indices] = True
        self.dot_grid[0, 0] = False  # 玩家起点无豆

        # 初始化墙壁（如果启用）
        self.wall_grid = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        if self.with_walls:
            # 示例墙壁：随机生成一些墙，确保玩家起点不是墙，且通路存在
            # 这里简单地在边界或内部生成一些墙，可根据需要设计
            # 为了可玩性，我们生成一些随机墙，但避免封闭区域
            num_walls = int(total_cells * 0.1)  # 10% 的墙
            wall_indices = np.random.choice(total_cells, num_walls, replace=False)
            self.wall_grid.flat[wall_indices] = True
            self.wall_grid[0, 0] = False  # 起点不能是墙

        # 初始化鬼魂位置（如果启用）
        self.ghost_positions = []
        if self.with_ghost:
            # 避免放在玩家起点和有墙的位置
            for _ in range(self.num_ghosts):
                while True:
                    gx = np.random.randint(0, self.grid_size)
                    gy = np.random.randint(0, self.grid_size)
                    if (gx, gy) != (0, 0) and not self.wall_grid[gx, gy]:
                        break
                self.ghost_positions.append([gx, gy])

        self.dots_left = np.sum(self.dot_grid)
        self.steps = 0
        self.ghost_collision = False
        return self._get_obs(),{}

    def _get_obs(self):
        # 玩家通道
        player = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        px, py = self.player_pos
        player[px, py] = 1.0

        # 鬼魂通道（可能多个鬼魂，每个位置置1）
        ghost = np.zeros_like(player)
        for gx, gy in self.ghost_positions:
            ghost[gx, gy] = 1.0

        # 墙壁通道
        wall = self.wall_grid.astype(np.float32)

        # 豆子通道
        dot = self.dot_grid.astype(np.float32)

        return np.stack([player, ghost, wall, dot], axis=0)

    def step(self, action):
        self.steps += 1
        reward = -0.1
        terminated = False
        truncated = False
        self.ghost_collision = False  # 先重置

        # 玩家移动逻辑（不变）
        x, y = self.player_pos
        new_x, new_y = x, y
        if action == 0:
            new_y = min(y + 1, self.grid_size - 1)
        elif action == 1:
            new_y = max(y - 1, 0)
        elif action == 2:
            new_x = max(x - 1, 0)
        elif action == 3:
            new_x = min(x + 1, self.grid_size - 1)
        if not self.wall_grid[new_x, new_y]:
            self.player_pos = [new_x, new_y]
            x, y = new_x, new_y

        # 吃豆逻辑（不变）
        if self.dot_grid[x, y]:
            self.dot_grid[x, y] = False
            self.dots_left -= 1
            reward += 1

        # 鬼魂移动及碰撞检测
        if self.with_ghost:
            new_ghost_positions = []
            for gx, gy in self.ghost_positions:
                dir = np.random.randint(4)
                ngx, ngy = gx, gy
                if dir == 0:
                    ngy = min(gy + 1, self.grid_size - 1)
                elif dir == 1:
                    ngy = max(gy - 1, 0)
                elif dir == 2:
                    ngx = max(gx - 1, 0)
                elif dir == 3:
                    ngx = min(gx + 1, self.grid_size - 1)
                if not self.wall_grid[ngx, ngy]:
                    new_ghost_positions.append([ngx, ngy])
                else:
                    new_ghost_positions.append([gx, gy])
            self.ghost_positions = new_ghost_positions

            # 检查碰撞
            for gx, gy in self.ghost_positions:
                if [x, y] == [gx, gy]:
                    reward -= 10
                    terminated = True          # 因撞鬼结束
                    self.ghost_collision = True
                    break

        # 检查是否吃完豆子
        if self.dots_left == 0:
            reward += 10
            terminated = True

        # 步数截断
        if self.steps >= 1000:
            truncated = True

        # 构建 info 字典
        info = {
            'ghost_collision': self.ghost_collision,
            'dots_left': self.dots_left,
            'steps': self.steps
        }

        return self._get_obs(), reward, terminated, truncated, info

    def render(self, mode='human'):
        # 简单文本渲染
        grid = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        # 标记豆子
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.dot_grid[i, j]:
                    grid[i][j] = 'o'
        # 标记墙壁
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.wall_grid[i, j]:
                    grid[i][j] = '#'
        # 标记鬼魂
        for gx, gy in self.ghost_positions:
            grid[gx][gy] = 'G'
        # 标记玩家
        px, py = self.player_pos
        grid[px][py] = 'P'
        # 打印
        for row in reversed(grid):
            print(' '.join(row))
        print(f"Steps: {self.steps}, Dots left: {self.dots_left}")

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor


# 1. 自定义特征提取器，适配 4x15x15 输入
class SmallGridFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # 输入形状: (4, 15, 15)
        n_input_channels = observation_space.shape[0]  # 4

        # 设计一个小型 CNN
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=1, padding=1),  # 输出: 32x15x15
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 输出: 32x7x7
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # 输出: 64x7x7
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 输出: 64x3x3
            nn.Flatten(),
        )

        # 计算扁平化后的特征维度
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        features = self.cnn(observations)
        return self.linear(features)

# 2. 环境工厂函数
def make_env():
    env = SimplePacmanEnv(grid_size=15, with_ghost=False, with_walls=False)
    env = Monitor(env)
    return env

# 3. 创建向量化环境
vec_env = DummyVecEnv([make_env])
eval_env = DummyVecEnv([make_env])

# 4. 评估回调（可选）
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./best_model",
    eval_freq=50000,
    n_eval_episodes=10,
)

# 5. 配置策略参数：使用自定义特征提取器 + 关闭归一化
policy_kwargs = dict(
    features_extractor_class=SmallGridFeatureExtractor,
    features_extractor_kwargs=dict(features_dim=256),
    normalize_images=False   # 因为你的观测已经是 [0,1] float32
)

# 6. 创建 PPO 模型
model = PPO(
    policy="CnnPolicy",
    env=vec_env,
    learning_rate=2.5e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log="./pacman_tensorboard/",
    policy_kwargs=policy_kwargs
)
# model=PPO.load("best_model1.zip",env=vec_env)
# 7. 开始训练
model.learn(total_timesteps=100_0000, callback=eval_callback,log_interval=200,reset_num_timesteps=False)
model.save("PPO.2.zip")
print("SAVE")

In [ ]:
# 测试训练好的模型（可选）
obs = vec_env.reset()
for step in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = vec_env.step(action)
    if dones[0]:
        obs = vec_env.reset()
        print("游戏结束")

In [ ]:
import matplotlib
print(matplotlib.__version__)
import matplotlib.pyplot as plt
print(plt.__file__)   # 看路径是哪个环境

In [ ]:
import sys
print(sys.executable)          # 确认是 Herta 环境的 python
import matplotlib
print(matplotlib.__version__)  # 应该 >= 3.3.0
print(matplotlib.__file__)     # 应该指向 Herta 环境下的 site-packages